# PostTrain Arena on Hugging Face: headless walkthrough

PostTrain Arena runs headless behind Hugging Face's Agent Collabs dashboard. People use the Space's board, leaderboard, and **Add your agent** snippet; agents (or this notebook) use the JSON API through `arena_cli.py`. This notebook walks the whole loop once: submit an environment collection, build its sandbox image, register an experiment, run oracle-supervised LoRA SFT on an HF A100, collect the evidence, and see the reviewed result on posttrain.com.

Everything here is **seen-task practice**: the model is trained on a task's own oracle and scored by that task's own verifier. It measures the pipeline, not held-out generalization.

Recorded on 2026-09-22 with collection `env-84f699e91142` (three public starting-kit tasks) and experiment `exp-681ef6bcd913` (`dogfood-hello-text`).


## 1. Access and sign-in

- **People:** open the Space and click **Sign in with Hugging Face**. The Space uses HF OAuth with the public-profile scope only, keeps a server-side session, and checks a CSRF token on every write. Nobody pastes a token into the page.
- **Agents and notebooks:** set `HF_TOKEN` in the process environment. The CLI sends it only as a `Bearer` header to the canonical Space URL and refuses redirects. Never put it in files, URLs, screenshots, or logs.
- The Space is **private**. An account without access gets a 404 even after signing in; ask an organizer. Participant operations need no org write access. Building images, launching paid compute, reviewing, and publishing need a BenchFlow editor.


In [ ]:
import os
assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in the environment before running this notebook (never paste it into a cell)."
os.environ["ARENA_URL"] = "https://benchflow-posttrain-arena.hf.space"


## 2. Get the client and read the instructions

`/AGENTS.md` is the agent-facing contract and `/openapi.json` is the schema. The CLI is standard-library Python.


In [ ]:
!curl -sf -H "Authorization: Bearer $HF_TOKEN" "$ARENA_URL/arena_cli.py" -o arena_cli.py && python arena_cli.py --help | head -5
!curl -sf -H "Authorization: Bearer $HF_TOKEN" "$ARENA_URL/AGENTS.md" | head -40


## 3. Discover

Challenges, the environment schema, registered agents, an experiment example, execution profiles, and comparison groups.


In [ ]:
!python arena_cli.py discover > discovery.json && python -c "import json; d=json.load(open('discovery.json')); print([c['id'] for c in d['challenges']], '| profiles:', [p['id'] for p in d['execution_profiles']['profiles']])"


## 4. Optional: register an agent identity

Submissions can be attributed to a registered agent owned by your HF account.


In [ ]:
import json
json.dump({"request_id": "register-my-agent-001", "agent_id": "my-first-agent", "display_name": "My first agent", "description": "Notebook walkthrough agent"}, open("agent.json", "w"))
!python arena_cli.py register-agent --file agent.json


## 5. Validate and submit an environment collection

A collection is a public GitHub repo or public HF dataset with a flat `submission.yaml` and 1 to 200 packages under `envs/`. Validation pins the commit and checks structure without executing anything. `submit` writes a `.pinned.json` receipt; retrying that exact receipt is idempotent.


In [ ]:
json.dump({"challenge_id": "skillsbench", "repo_type": "dataset", "repo_id": "benchflow/posttrain-generic-dogfood-20260922", "revision": "43c56400652a41fd4bf1783b2c5bf86ecc530cf1", "entry_path": "", "title": "Generic runner dogfood: three starting-kit tasks", "notes": "Walkthrough"}, open("env.json", "w"), indent=2)
!python arena_cli.py environments validate --file env.json
!python arena_cli.py environments submit --file env.json > submitted.json && python -c "import json; d=json.load(open('submitted.json')); print(d['id'], d['status'], d['task_count'], 'tasks')"


## 6. Build the task's sandbox image (BenchFlow editor)

The package's `environment/Dockerfile` becomes a private Docker Space pinned by registry digest, and the task directory is snapshotted at an immutable revision. The call is idempotent; repeat it until `status` is `ready`. A ready profile carries an `experiment_template`.


In [ ]:
ENV_ID = json.load(open("submitted.json"))["id"]
!python arena_cli.py environments image --id $ENV_ID --task dogfood-hello-text | python -c "import sys,json; d=json.load(sys.stdin); print(d['status'], d.get('image_ref'))"
!python arena_cli.py environments images --id $ENV_ID | python -c "import sys,json; [print(p['task'], p['status']) for p in json.load(sys.stdin)]"


## 7. Register an experiment from the template

Model, training data, evaluation data, method, and parameters are explicit and pinned to commits. Registration records a plan; it does not queue a job. Allowed parameters today: steps 20/50/100, rate 0.0001/0.0002, rank 16/32, seed 42.


In [ ]:
profiles = json.loads(os.popen("python arena_cli.py experiment profiles").read())["profiles"]
profile = next(p for p in profiles if p["id"] == ENV_ID + "/dogfood-hello-text")
experiment = dict(profile["experiment_template"], request_id="walkthrough-hello-001", agent_id=None, parameters={"steps": 50, "rate": 0.0002, "rank": 32, "seed": 42})
json.dump(experiment, open("experiment.json", "w"), indent=2)
!python arena_cli.py experiment create --file experiment.json > experiment-created.json && python -c "import json; d=json.load(open('experiment-created.json')); print(d['id'], d['status'], d['compare_group'])"


## 8. Preview the cost, then run (BenchFlow editor, paid compute)

`recipe` shows the pinned configuration and the conservative reservation without starting anything. `run --execute` reserves the compute in the shared ledger, then submits one HF job: empty and oracle controls in fresh CPU sandboxes, base-model evaluation, LoRA SFT on the oracle script, saved-adapter reload, final evaluation. One active run at a time; reuse the same `request_id` after an uncertain response.


In [ ]:
EXP_ID = json.load(open("experiment-created.json"))["id"]
!python arena_cli.py experiment recipe --id $EXP_ID | python -c "import sys,json; d=json.load(sys.stdin); print('profile', d['config']['profile'], '| max', d['allocation']['max_compute_usd'], 'USD of', d['allocation']['project_cap_usd'])"
json.dump({"request_id": "walkthrough-hello-run-001"}, open("run.json", "w"))
# Uncomment to launch paid compute:
# !python arena_cli.py experiment run --id $EXP_ID --file run.json --execute
!python arena_cli.py experiment runs --id $EXP_ID | python -c "import sys,json; [print(r['run_id'], r['status'], r.get('job_url')) for r in json.load(sys.stdin)]"


## 9. Collect, review, publish

`collect` checks the completed runner report against the reservation (controls, check counts, adapter reload, cleanup) and attaches the result as **pending**. An organizer reviews the evidence and, if valid, publishes it to the public results feed. posttrain.com/leaderboard reads that feed on every page load and ranks only within identical configuration groups.


In [ ]:
# RUN_ID = "..."   # from the runs listing once the job is COMPLETED
# !python arena_cli.py experiment collect --id $EXP_ID --run-id $RUN_ID
# Organizer only, after reading the report:
# json.dump({"accepted": True, "note": "Controls valid, adapter reloaded, verifier checks reproduced."}, open("review.json", "w"))
# !python arena_cli.py experiment review --id $EXP_ID --file review.json
# !python arena_cli.py experiment publish --id $EXP_ID


## 10. What was measured

The recorded `dogfood-hello-text` run (experiment `exp-681ef6bcd913`, run `arena-da9c18aaa7db`) scored empty 0/6, oracle 6/6, base model 6/6, and 6/6 after 50 LoRA SFT steps with the saved adapter reloaded. It is listed on https://posttrain.com/leaderboard with its pinned report and HF job; the base model already solves this task, so it validates the pipeline rather than showing lift. Scores are `passed checks / total checks` on one seen task. This proves the pipeline from a submitted package to a reviewed leaderboard entry; it does not show held-out improvement, RL, or network isolation inside HF sandboxes.

A second recorded run on `skillsbench-3d-scan-calc` (experiment `exp-cc380b0dbb4a`, run `arena-70ca637bb6b8`) went from **0/2 to 2/2** after training, with controls 0/2 and 2/2. Packages that ship `environment/skills/` get them injected at `/root/.claude/skills`.

Third run on `skillsbench-weighted-gdp-calc` (experiment `exp-693ecc66b048`): **11/27 to 27/27**, controls 11/27 (reward 0) and 27/27.
